# Exploration Testset

Exploring the format of the data and the files

In [ ]:
import numpy as np 
import cbor2 
from pathlib import Path
import pandas as pd 

base = Path.cwd() 
target = base / "../Testset/benchmarkY2test-auto-manual-qrels/benchmarkY2test"

!tree -L 1 {target}

/tempory/the_three_potatoes/ri_project/workspaces/amelie/RI_Project/Notebooks/../Testset/benchmarkY2test-auto-manual-qrels/benchmarkY2test
├── benchmarkY2test-entity-automatic.qrels
├── benchmarkY2test-entity-lenient.qrels
├── benchmarkY2test-entity-manual.qrels
├── benchmarkY2test-entity-with-passage-manual.qrels
├── benchmarkY2test-goldarticles.cbor
├── benchmarkY2test-goldpassages.cbor
├── benchmarkY2test-goldpassages.qrels
├── benchmarkY2test-psg-lenient.qrels
├── benchmarkY2test-psg-manual.qrels
├── interface.json
└── README.mkd

1 directory, 11 files


---
source : https://trec-car.cs.unh.edu/

The **TREC CAR Y2** has 2 tasks:
- passage ranking
- entity ranking 

The evaluation is as follow: given a query Q, for each of its sections (headers ?), retrieve a ranking of relevant entity-passage tuple (E, P).
Passage come from the corpus, entity is the entry to the knowledge base. 
We define a passage or entity as relevant if the passage content or entity should be mentioned in the knowledge article. Different degrees of “could be mentioned”, “should be mentioned”, or “must be mentioned” are represented through a graded annotation scale during manual assessment.

In the passage ranking task, entities are omitted.

In the entity ranking task, the entity must be given, and optionally, complemented with a passage that serves as provenance for why the entity is relevant. If the passage is omitted, it will be replaced with the first paragraph from the entity’s article.

---

Qrels files correspond to the annotations of the train and test data.
They follow this format

TOPIC |      ITERATION |      DOCUMENT# |      RELEVANCY |

where TOPIC is the topic number,

ITERATION is the feedback iteration (almost always zero and not used),

DOCUMENT# is the official document number that corresponds to the "docno" field in the documents, and

RELEVANCY is a binary code of 0 for not relevant and 1 for relevant.



---
We explore some of the files. As stated in the article, only the automatic judgement is considered.

#### benchmarkY2test-entity-automatic.qrels

In [ ]:
entity_auto_qrels = pd.read_csv(
    target / "benchmarkY2test-entity-automatic.qrels",
    sep=r"\s+",
    header=None,
    names=["query_id", "iteration", "doc_id", "relevance"]
)

entity_auto_qrels.drop(columns=['iteration'], inplace=True) # iteration is always 0, so we can drop it
entity_auto_qrels.head()

# for each query_id, the docs that are relevant  

,query_id,doc_id,relevance
0,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Lactic%20acid,1
1,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Phenotype,1
2,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Biological%20engineering,1
3,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Cytochrome%20c%20oxidase,1
4,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Redox,1


In [20]:
entity_auto_qrels.shape, entity_auto_qrels['relevance'].unique()

((17044, 3), array([1]))

In [28]:
entity_auto_qrels.head()['query_id'].unique(), entity_auto_qrels['query_id'].nunique()

(<StringArray>
 ['enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species']
 Length: 1, dtype: str,
 976)

#### benchmarkY2test-entity-lenient.qrels (not taken into account as not automatic)

In [23]:
entity_len_qrels = pd.read_csv(
    target / "benchmarkY2test-entity-lenient.qrels",
    sep=r"\s+",
    header=None,
    names=["query_id", "iteration", "doc_id", "relevance"]
)

entity_len_qrels.drop(columns=['iteration'], inplace=True) # iteration is always 0, so we can drop it
entity_len_qrels.head()

# for each query_id, the docs that are relevant  

,query_id,doc_id,relevance
0,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Anaerobic%20respiration,0
1,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Biohydrogen,0
2,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Carbon%20dioxide,2
3,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Cellular%20respiration,2
4,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,enwiki:Citric%20acid%20cycle,2


In [29]:
entity_len_qrels.shape, sorted(entity_len_qrels['relevance'].unique())

((8415, 3),
 [np.int64(-2),
  np.int64(0),
  np.int64(2),
  np.int64(3),
  np.int64(4),
  np.int64(5)])

In [27]:
entity_len_qrels.head()['query_id'].unique(), entity_len_qrels['query_id'].nunique()

(<StringArray>
 ['enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/E.%20coli%20mutant']
 Length: 1, dtype: str,
 271)

#### benchmarkY2test-entity-with-passage-manual.qrels (not used)

In [30]:
tmp = pd.read_csv(
    target / "benchmarkY2test-entity-with-passage-manual.qrels",
    sep=r"\s+",
    header=None,
    names=["query_id", "iteration", "doc_id", "relevance"]
)

tmp.drop(columns=['iteration'], inplace=True) # iteration is always 0, so we can drop it
tmp.head()

# for each query_id, the docs that are relevant  

,query_id,doc_id,relevance
0,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,00de04a88d0f50387f488c9fb8f71ac24a4b0048/enwik...,0
1,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,0ba471e24cf7b3aa7b129c9148fefc891c39e337/enwik...,-1
2,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,1d18136a6764db7de09cabd4ef8a2852980ce275/enwik...,0
3,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,20558c26a8282f9787de765f331c01df8b9a0ecc/enwik...,-1
4,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,296f175bc90acda15321b5a91ef81a20e9faa5e1/enwik...,0


In [31]:
tmp.shape, sorted(tmp['relevance'].unique())

((11440, 3),
 [np.int64(-2),
  np.int64(-1),
  np.int64(0),
  np.int64(1),
  np.int64(2),
  np.int64(3)])

#### benchmarkY2test-goldpassages.qrels

In [ ]:
gold_psg_qrels = pd.read_csv(
    target / "benchmarkY2test-goldpassages.qrels",
    sep=r"\s+",
    header=None,
    names=["query_id", "iteration", "doc_id", "relevance"]
)

gold_psg_qrels.drop(columns=['iteration'], inplace=True) # iteration is always 0, so we can drop it
gold_psg_qrels.head()

# for each query_id, the passages that are relevant  

,query_id,doc_id,relevance
0,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,559480f166353af56ab3ba23397aa1f246425a0c,1
1,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,6d7a5f2baf661a86cec7d56fc08e079e34868a42,1
2,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,a9f022e5fddd42ede054fc5b9142d4b4586217f7,1
3,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,f687899c8c33d513212a7d0d67a247d8e8917b80,1
4,enwiki:Aerobic%20fermentation/Aerobic%20fermen...,a9f022e5fddd42ede054fc5b9142d4b4586217f7,1


In [33]:
gold_psg_qrels.shape, sorted(gold_psg_qrels['relevance'].unique())

((3598, 3), [np.int64(1)])

In [34]:
gold_psg_qrels.head()['query_id'].unique(), gold_psg_qrels['query_id'].nunique()

(<StringArray>
 ['enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species', 'enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/E.%20coli%20mutant']
 Length: 2, dtype: str,
 1002)

#### goldarticles.cbor

In [35]:
gold_art = []

with open(target / "benchmarkY2test-goldarticles.cbor", "rb") as f:
    for i in range(2): # only 2 objects !
        obj = cbor2.load(f)
        gold_art.append(obj)
        print(type(obj))
        print(obj)
        print("=" * 50) 

<class 'list'>
['CAR', [0], [0, [[0, 'tqa', 'en-us', 'tqa', []], [0, 'enwiki', 'en-US', '20180601', []]], 'benchmarkY2test', [], []]]
<class 'list'>
[[0, 'relative ages of rocks', b'tqa:relative%20ages%20of%20rocks', [[0, 'Matching Rock Layers', b'Matching%20Rock%20Layers', [[1, [0, b'-5811015104952104000', [[0, 'When rock layers are in the same place, its easy to give them relative ages. But what if rock layers are far apart? What if they are on different continents? What evidence is used to match rock layers in different places? ']]]]]], [0, 'Putting Events in Order', b'Putting%20Events%20in%20Order', [[1, [0, b'8537861184964954752', [[0, 'To create the geologic time scale, geologists correlated rock layers. Stenos laws were used to determine the relative ages of rocks. Older rocks are at the bottom and younger rocks are at the top. The early geologic time scale could only show the order of events. The discovery of radioactivity in the late 1800s changed that. Scientists could determ

In [36]:
# forma first element
def parse_car_header(obj):
    return {
        "format": obj[0],             
        "version": obj[1][0],         
        "corpora": [
            {
                "id": x[1],
                "lang": x[2],
                "source": x[3]
            }
            for x in obj[2][1]
        ],
        "dataset": obj[2][2]
    }

parse_car_header(gold_art[0])

{'format': 'CAR',
 'version': 0,
 'corpora': [{'id': 'tqa', 'lang': 'en-us', 'source': 'tqa'},
  {'id': 'enwiki', 'lang': 'en-US', 'source': '20180601'}],
 'dataset': 'benchmarkY2test'}

In [42]:
gold_art[1][10]

[0,
 'renewable energy resources',
 b'tqa:renewable%20energy%20resources',
 [[0,
   'Biomass',
   b'Biomass',
   [[1,
     [0,
      b'757503345757400822',
      [[0,
        'Biomass is another renewable source of energy. Biomass includes wood, grains, and other plant materials or waste materials. People can burn wood directly for energy in the form of heat. Biomass can also be processed to make biofuel. Biofuel is a fairly new type of energy that is becoming more popular. Biomass is useful because it can be made liquid. This means that they can be used in cars and trucks. Some car engines can be powered by pure vegetable oil or even recycled vegetable oil. Sometimes the exhaust from these cars smells like French fries! By using biofuels, we can cut down on the amount of fossil fuel that we use. Because living plants take carbon dioxide out of the air, growing plants for biofuel can mean that we will put less of this gas into the air overall. This could help us do something about the 

In [37]:
def parse_topic(node):
    """
    Recursively parse CAR topic tree
    """
    level = node[0]
    title = node[1]
    topic_id = node[2].decode("utf-8") if isinstance(node[2], bytes) else node[2]

    children = []

    # node[3] contains subtopics
    for child in node[3]:
        children.append(parse_topic(child))

    return {
        "level": level,
        "title": title,
        "id": topic_id,
        "children": children
    }

def parse_all_topics(raw_topics):
    return [parse_topic(t) for t in raw_topics]

In [38]:
topics_clean = parse_all_topics(gold_art[1])  # second list item
topics_clean

IndexError: list index out of range

#### goldpassage.cbor

In [45]:
gold_passages = []

with open(target / "benchmarkY2test-goldpassages.cbor", "rb") as f:
    for i in range(2): # only 2 objects !
        obj = cbor2.load(f)
        gold_passages.append(obj)
        print(type(obj))
        print(obj)
        print("=" * 50) 

<class 'list'>
['CAR', [2], [0, [[0, 'tqa', 'en-us', 'tqa', []], [0, 'enwiki', 'en-US', '20180601', []]], 'benchmarkY2test', [], []]]
<class 'list'>
[[0, b'-1029561825483581544', [[0, 'La Nia generally follows El Nio. It occurs when the Pacific Ocean is cooler than normal. Figure 17.26 shows what happens. The trade winds are like they are in a normal year. They blow from east to west. But in a La Nia the winds are stronger than usual. More cool water builds up in the western Pacific. These changes can also affect climates worldwide. ']]], [0, b'-1063658338102941709', [[0, 'Many of the glaciers in Glacier National Park have shrunk and are no longer active. Summer temperatures have risen rapidly in this part of the country and so the rate of melting has picked up. Whereas Glacier National Park had 150 glaciers in 1850, there are only about 25 today. Recent estimates are that the park will have no active glaciers as early as 2020. This satellite image shows Grinnell Glacier, Swiftcurrent 

In [46]:
parse_car_header(gold_passages[0])

{'format': 'CAR',
 'version': 2,
 'corpora': [{'id': 'tqa', 'lang': 'en-us', 'source': 'tqa'},
  {'id': 'enwiki', 'lang': 'en-US', 'source': '20180601'}],
 'dataset': 'benchmarkY2test'}